# Suite EXT — Schema extension

A schema is extended by adding members to it: properties and adjacencies to an object schema, properties to a
relation, branches to a union ([CAP-116: framework schemas extended, checked](../../../mbse-basis/docs/capabilities/CAP-116.md)). A module extends schemas in its `install(txn)`, a transaction
the store runs (object identity 86 in [CAP-115: schemas defined by idempotent installs](../../../mbse-basis/docs/capabilities/CAP-115.md#decided); [CAP-115: schemas defined by idempotent installs](../../../mbse-basis/docs/capabilities/CAP-115.md)). A schema the store was made holding, a
framework schema, only gains members, in any transaction; a store's own schemas change freely. Each store's framework
schemas are its own, so what one store's install adds, another lacks.

In [ ]:
import { Builders as B, Paths, Plain, Validators } from "@mbse/schemas/Framework";
import { assert, entries_in, map, raises, same, spec_of } from "./support.js";

/** crm.Person, who knows persons and has tags; crm.Customer, refining it; crm.Vip, refining that; crm.Id, a union. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Knows: S.OfRelation, Person: S.OfObject, Customer: S.OfObject, Vip: S.OfObject, Id: S.OfUnion })
    .spec([
      (_: any) => _.Knows.name("crm.Knows").links([(t: any) => t.name("a"), (t: any) => t.name("b")]).uniques([]), // a set
      (_: any) => _.Person.name("crm.Person").ref(true).properties({
        name: S.OfStr, tags: S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfStr])) }).adjacencies({
        knows: (a: any) => a.relation([_.Knows]).me("a") }),
      (_: any) => _.Customer.name("crm.Customer").ref(true).refines([_.Person]).properties({ account: S.OfStr }),
      (_: any) => _.Vip.name("crm.Vip").ref(true).refines([_.Customer]).properties({ tier: S.OfInt }).adjacencies({
        fans: (a: any) => a.relation([_.Knows]).me("b") }), // a refinement of a refinement
      (_: any) => _.Id.name("crm.Id").branches([(t: any) => t.name("n").schema([S.OfInt])]),
    ])]))]);
}

const store: any = new B.Store();
store.transaction(crm);
const Text = store.Schemas.OfStr.as_schema(); // the store's own
const Int = store.Schemas.OfInt.as_schema(); // the store's own
const [Knows, Person, Customer, Id] = ["crm.Knows", "crm.Person", "crm.Customer", "crm.Id"].map((name) => store.registered(name));
const EXTENDS = "an extension only adds members to";

/** crm.Person's age and an anonymous end of crm.Knows, crm.Knows' since, crm.Id's text branch, and crm.Note. */
function more(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("crm.more/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Person: S.OfObject, Knows: S.OfRelation, Id: S.OfUnion, Note: S.OfObject })
    .spec([
      (_: any) => _.Person.name("crm.Person").properties({ age: S.OfInt }).adjacencies([
        (a: any) => a.relation([_.Knows]).me("b")]), // anonymous
      (_: any) => _.Knows.name("crm.Knows").properties({ since: S.OfInt }),
      (_: any) => _.Id.name("crm.Id").branches([(t: any) => t.name("s").schema([S.OfStr])]),
      (_: any) => _.Note.name("crm.Note").properties({ text: S.OfStr }), // built: its own
    ])]))]);
}

/** Domains, a scalar's domain and a quantity's unit: what mbse-expressions adds to a store, which extends its own
 * `Schemas.OfScalar` and `Schemas.OfObject`. */
function domains(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("domains/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Domain: S.OfObject, HasDomain: S.OfRelation, Scalar: S.OfObject, Object: S.OfObject })
    .spec([
      (_: any) => _.HasDomain.name("Expressions.Framework.HasDomain").links([
        (t: any) => t.name("schema"), (t: any) => t.name("domain")]).uniques([["schema"]]),
      (_: any) => _.Domain.name("Expressions.Framework.Domain").ref(true).properties({ label: S.OfStr }).adjacencies({
        schemas: (a: any) => a.relation([_.HasDomain]).me("domain") }),
      (_: any) => _.Scalar.name("Schemas.OfScalar").adjacencies({ domain: (a: any) => a.relation([_.HasDomain]).me("schema") }),
      (_: any) => _.Object.name("Schemas.OfObject").properties({ unit: S.OfStr }),
    ])]))]);
}

## EXT-01 · An install adds members to the schemas a store holds; their objects and their refinements gain them

In [ ]:
{
  const [ada] = store.transaction(() => {
    const ada = store.crm.Person((p: any) => p.name("Ada")); // built before the install
    return [ada];
  });
  store.transaction(more);
  assert(same([...Person.properties.keys()], ["name", "tags", "age"]) && Person.anonymous.length === 1);
  assert(same([...Customer.properties.keys()], ["name", "tags", "age", "account"])); // inherited
  assert(same([...Knows.entry.properties.keys()], ["since"]) && same(Id.branches.map((b: any) => b.name), ["n", "s"]));
  store.transaction(() => {
    assert(ada.age.get() === null && store.crm.Person(ada, (p: any) => p.age(36n)).age.get() === 36n); // absent until written
    assert(store.crm.Customer((c: any) => c.name("Bo").age(7n).account("A")).age.get() === 7n);
  });
  assert(store.registered("crm.Note").validate().length === 0);
  store.transaction(more); // run again: found, nothing written
  assert(same([...Person.properties.keys()], ["name", "tags", "age"]) && Person.anonymous.length === 1);
}

## EXT-02 · A schema the store was made holding only gains members, each under a name it neither has nor a refinement of it declares; the extension check says what else a write would change

In [ ]:
{
  const [Kind] = store.transaction(() => {
    const Kind = store.Schemas.OfObject((o: any) => ( // a store's own schema refining a framework schema
      o
        .name("crm.Kind")
        .refines([store.Schemas.OfObject])
        .properties([(t: any) => t.name("badge").schema([Text])])
    ));
    store.Schemas.OfObject((o: any) => ( // and a refinement of that
      o
        .name("crm.SubKind")
        .refines([Kind])
        .adjacencies([(t: any) => t.name("pins").relation([Knows]).me("a")])
    ));
    return [Kind];
  });
  const [OfObject, HasSchema] = [store.Schemas.OfObject.as_schema(), store.Schemas.HasSchema.as_schema()];
  for (const [change, says] of [
    [() => store.Schemas.OfObject(OfObject, (o: any) => o.description("An object schema")),
      "'Schemas.OfObject': it changes 'description'"],
    [() => store.Schemas.OfObject(OfObject, (o: any) => o.ref(false)), "'Schemas.OfObject': it changes 'ref'"],
    [() => store.Schemas.OfObject(OfObject, (o: any) => o.properties((adv: any) => adv.remove((e: any) => e.name("ref")))),
      "'Schemas.OfObject': it removes or replaces what 'properties' holds"],
    [() => store.Schemas.OfObject(OfObject, (o: any) => o.properties([(t: any) => t.name("ref").schema([Int])])), // retyped
      "'Schemas.OfObject': it removes or replaces what 'properties' holds"],
    [() => store.Schemas.OfRelation(HasSchema, (r: any) => r.links([(t: any) => t.name("c")])),
      "'Schemas.HasSchema': it adds to 'links', which an extension cannot"],
    [() => store.Schemas.OfObject(OfObject, (o: any) => o.adjacencies([(t: any) => t.name("ref").relation([Knows]).me("a")])),
      "'Schemas.OfObject': it adds 'ref', which it has"],
    [() => store.Schemas.OfObject(OfObject, (o: any) => o.properties([(t: any) => t.name("badge").schema([Text])])),
      "'Schemas.OfObject': it adds 'badge', which 'crm.Kind' declares"],
    [() => store.Schemas.OfObject(OfObject, (o: any) => o.properties([(t: any) => t.name("pins").schema([Text])])),
      "'Schemas.OfObject': it adds 'pins', which 'crm.SubKind' declares"],
    [() => store.Schemas.OfContainer(store.Schemas.OfRelation.as_schema().properties.get("uniques").schema,
      (c: any) => c.layout("sparse")), // unnamed: its kind
    "a container schema: it changes 'layout'"],
  ] as [() => unknown, string][]) {
    raises(TypeError, () => store.transaction(() => change()), `${EXTENDS} ${says}`);
  }
  assert(Kind !== null && !OfObject.properties.has("badge") && OfObject.ref && OfObject.description === null); // refused: unwritten
  store.transaction(() => {
    store.Schemas.OfObject(Person, (o: any) => o.description("A person").ref(false)); // a store's own schema changes freely
  });
  assert(Person.description === "A person" && !Person.ref);
}

## EXT-03 · A framework schema gains members in any transaction, an imperative scope's or an install's, and changes no other way

In [ ]:
{
  store.transaction(() => {
    store.Schemas.OfObject(store.Schemas.OfScalar, (o: any) => o.properties([(t: any) => t.name("unit").schema([Text])]));
    store.Schemas.OfObject(store.Schemas.OfScalar, (o: any) => o.adjacencies([(t: any) => t.relation([Knows]).me("b")])); // anonymous
    raises(TypeError, () => store.Schemas.OfObject(store.Schemas.OfScalar, (o: any) => o.description("x")),
      `${EXTENDS} 'Schemas.OfScalar': it changes 'description'`);
  });
  assert(store.Schemas.OfScalar.as_schema().properties.has("unit") && store.Schemas.OfScalar.as_schema().description === null);
  assert(store.Schemas.OfScalar.as_schema().anonymous.at(-1).relation === Knows);
  assert(new B.OfScalar.Builder(store.Schemas.OfInt, false, store).unit.get() === null); // a builder of its schemas has it
}

## EXT-04 · What an install adds to a framework schema is its store's alone: another store holds its own; a meta-schema's new members are its schemas' builders'

In [ ]:
const [extended, plain]: any[] = [new B.Store(), new B.Store()];
extended.transaction(domains);
assert(extended.Schemas.OfScalar.as_schema().adjacencies.has("domain") && extended.Schemas.OfObject.as_schema().properties.has("unit"));
assert(!plain.Schemas.OfScalar.as_schema().adjacencies.has("domain") && !plain.names().includes("Expressions.Framework.Domain"));
assert(!extended.framework(extended.registered("Expressions.Framework.Domain"))); // what it builds is the store's own
const [Meters] = extended.transaction(() => {
  const Meters = extended.Schemas.OfObject((o: any) => (
    o
      .name("Meters")
      .properties([(t: any) => t.name("value").schema([extended.Schemas.OfFloat])])
      .unit("m")
  ));
  return [Meters];
});
assert(new B.OfObject.Builder(Meters, false, extended).unit.get() === "m");
const [Feet] = extended.transaction(() => {
  const Feet = new B.OfObject.Builder(Meters, true, extended).name("Feet").clone();
  return [Feet];
});
assert(new B.OfObject.Builder(Feet, false, extended).unit.get() === "m"); // a clone keeps it
assert(new B.OfObject.Builder(plain.Schemas.OfObject, false, plain).unit.get() === null); // another store's builders lack it
assert(new B.OfScalar.Builder(plain.Schemas.OfInt, false, plain).domain.get() === null);

function trio(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("trio/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ T3: S.OfRelation, Scalar: S.OfObject })
    .spec([
      (_: any) => _.T3.name("T3").links([(t: any) => t.name("a"), (t: any) => t.name("b"), (t: any) => t.name("c")]),
      (_: any) => _.Scalar.name("Schemas.OfScalar").adjacencies({ trio: (a: any) => a.relation([_.T3]).me("a") }),
    ])]))]);
}

raises(TypeError, () => (new B.Store() as any).transaction(trio),
  "an extension's adjacency 'trio' is to a relation of two links and no properties, got 'T3'");

## EXT-05 · An extension's relation links schema objects and data: one entry, held by both, written from either end and once in a snapshot; what a store lacking the extension ignores

In [ ]:
{
  const s: any = new B.Store();
  s.transaction(domains);
  const [int8, Byte] = s.transaction(() => {
    const int8 = s.Expressions.Framework.Domain((d: any) => d.label("int8"));
    const Byte = s.Schemas.OfScalar(s.Schemas.OfInt, (x: any) => x.domain([int8])); // from the schema's end: the store's own scalar
    assert(same([...int8.schemas].map((e: any) => e.schema === s.Schemas.OfInt.as_schema()), [true])); // within the transaction too
    return [int8, Byte];
  });
  assert(Byte === s.Schemas.OfInt.as_schema() && same([...int8.schemas].map((e: any) => e.schema === Byte), [true])); // seen from the data's end
  const [int16] = s.transaction(() => {
    const int16 = s.Expressions.Framework.Domain((d: any) => d.label("int16").schemas((a: any) => a.insert((e: any) => e.schema(Byte))));
    return [int16];
  });
  assert(same([...new B.OfScalar.Builder(Byte, false, s).domain].map((e: any) => e.domain.label.get()), ["int8", "int16"]));
  assert(Validators.Validate(s).Reachable(s.registered("Expressions.Framework.Domain"), int8).length === 0);
  const text = (value: unknown) => JSON.stringify(value, (_k, v) => (v instanceof Map ? Object.fromEntries(v) : typeof v === "bigint" ? `${v}n` : v));
  const log = Plain.ToPlain(s).Snapshot() as any[];
  const held = log[1].get("mutations")[0].get("upsert").get("exists")[0].get("symbols"); // the data, and the scalar its entries link
  const written = [...held.keys()].flatMap((symbol: string) => ["schemas", "domain"].flatMap((name) => entries_in(spec_of(log, symbol, 1), name)));
  assert(written.length === 2 && written.every((e) => e.has("schema"))); // each once, from the data's end
  const back: any = new B.Store(); // a round trip between stores with the same extensions
  back.transaction(domains);
  Plain.FromPlain(back).Snapshot(log);
  assert([...new B.OfScalar.Builder(back.Schemas.OfInt, false, back).domain].length === 2);
  assert(text(Plain.ToPlain(back).Snapshot()) === text(log));
  s.transaction(() => {
    s.Schemas.OfScalar(Byte, (x: any) => x.domain((a: any) => a.remove((e: any) => e.domain(int8))));
  });
  assert([...int8.schemas].length === 0 && [...int16.schemas].length === 1);
  // A store's basic scalar is its own: the entries linking it are the store's, and another store's is never linked.
  const [one, other]: any[] = [new B.Store(), new B.Store()];
  one.transaction(domains);
  other.transaction(domains);
  one.transaction(() => {
    one.Expressions.Framework.Domain((d: any) => d.label("machine int").schemas((a: any) => a.insert((e: any) => e.schema(one.Schemas.OfFloat))));
  });
  const seen = (store: any) => [...new B.OfScalar.Builder(store.Schemas.OfFloat, false, store).domain].map((e: any) => e.domain.label.get());
  assert(same(seen(one), ["machine int"]) && same(seen(other), []));
  const third: any = new B.Store();
  third.transaction(domains);
  Plain.FromPlain(third).Snapshot(Plain.ToPlain(one).Snapshot());
  assert(same(seen(third), ["machine int"]) && same(seen(other), []));
  assert(text(Plain.ToPlain(third).Snapshot()) === text(Plain.ToPlain(one).Snapshot())); // read back as written
  one.transaction(() => {
    raises(TypeError, () => one.Expressions.Framework.Domain((d: any) => d.schemas((a: any) => a.insert((e: any) => e.schema(
      other.Schemas.OfInt)))), "proxies can only link proxy instances, or schema objects their store holds");
  });
  // Read into a store without an extension, a sub-clause whose meaning needs it is ignored, and kept where asked: a
  // property a schema the store was made holding lacks, an entry only such a schema could hold, a property of an entry of
  // such a relation. The rest is read.
  const bare: any = new B.Store();
  let ignored: Plain.Ignored[] = [];
  Plain.FromPlain(bare).Snapshot(Plain.ToPlain(extended).Snapshot(), ignored);
  assert(same(ignored.map((i) => i.reason), ["Schemas.OfObject has no property 'unit'", "Schemas.OfObject has no property 'unit'"])); // Meters', Feet's
  assert(ignored.every((i) => i.path.endsWith(".as_property")) && same([...bare.registered("Meters").properties.keys()], ["value"]));
  /** A tag a scalar has, declared from the scalar's end only. */
  const tagged = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("tagged/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Tags: S.OfRelation, Tag: S.OfObject, Scalar: S.OfObject })
      .spec([
        (_: any) => _.Tags.name("X.Tags").links([(t: any) => t.name("scalar"), (t: any) => t.name("tag")]),
        (_: any) => _.Tag.name("X.Tag").ref(true),
        (_: any) => _.Scalar.name("Schemas.OfScalar").adjacencies({ tags: (a: any) => a.relation([_.Tags]).me("scalar") }),
      ])]))]);
  };
  const labels: any = new B.Store();
  labels.transaction(tagged);
  labels.transaction(() => {
    labels.Schemas.OfScalar(labels.Schemas.OfInt, (x: any) => x.tags([labels.X.Tag((x: any) => x)]));
    labels.Schemas.OfObject((o: any) => o.name("Note").properties([(t: any) => t.name("text").schema([labels.Schemas.OfStr])]));
  });
  ignored = [];
  Plain.FromPlain(new B.Store()).Snapshot(Plain.ToPlain(labels).Snapshot(), ignored);
  assert(same(ignored.map((i) => i.reason), ["Schemas.OfScalar has no adjacency 'tags'"])); // written from the scalar's end
  const altered = Plain.ToPlain(labels).Snapshot() as any[]; // as an extension adding a property to Schemas.HasSchema would write it
  const entry = spec_of(altered, "Note").find((m) => m.has("as_adjacency")).get("as_adjacency").get("mutations")[0].get("insert").get("entries")[0];
  const typing = entry.get("mutations").find((p: any) => p.has("as_link")).get("as_link").get("mutations")[0].get("set").get("def").get("mutations");
  const typed = typing.find((m: any) => m.has("as_adjacency") && m.get("as_adjacency").get("name") === "schema");
  typed.get("as_adjacency").get("mutations")[0].get("insert").get("entries")[0].get("mutations").push(
    map({ as_property: { name: "why", mutations: [{ set: { value: "sized" } }] } }));
  ignored = [];
  Plain.FromPlain(new B.Store()).Snapshot(altered, ignored);
  assert(same(ignored.map((i) => i.reason), [
    "an entry of 'schema' has no property 'why'", "Schemas.OfScalar has no adjacency 'tags'"]));
  assert(Plain.FromPlain(new B.Store()).Snapshot(altered).length > 0); // ignored, kept or not

  /** A scalar's shapes, object schemas, declared from the scalar's end only. */
  const shaped = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("shaped/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Shapes: S.OfRelation, Scalar: S.OfObject })
      .spec([
        (_: any) => _.Shapes.name("X.Shapes").links([(t: any) => t.name("scalar"), (t: any) => t.name("shape")]),
        (_: any) => _.Scalar.name("Schemas.OfScalar").adjacencies({ shapes: (a: any) => a.relation([_.Shapes]).me("scalar") }),
      ])]))]);
  };
  // An entry written from a store's own scalar, which a snapshot never writes, is written still, and what it links that
  // has no name of its own follows it.
  const shapes: any = new B.Store();
  shapes.transaction(shaped);
  shapes.transaction(() => {
    const shape = shapes.Schemas.OfObject((o: any) => o.properties([(t: any) => t.name("x").schema([shapes.Schemas.OfInt])])); // unnamed
    shapes.Schemas.OfScalar(shapes.Schemas.OfInt, (x: any) => x.shapes([shape]));
  });
  const shaping = Plain.ToPlain(shapes).Snapshot();
  const again: any = new B.Store();
  again.transaction(shaped);
  Plain.FromPlain(again).Snapshot(shaping);
  assert([...new B.OfScalar.Builder(again.Schemas.OfInt, false, again).shapes].length === 1
    && text(Plain.ToPlain(again).Snapshot()) === text(shaping));
  assert(Plain.FromPlain(again).Snapshot(shaping).length === 0 && [...new B.OfScalar.Builder(again.Schemas.OfInt, false, again).shapes].length === 1);

  /** A board, a singleton, pinning schemas: data that links schema objects. */
  const pinned = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("pinned/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Pins: S.OfRelation, Board: S.OfObject })
      .spec([
        (_: any) => _.Pins.name("X.Pins").links([(t: any) => t.name("board"), (t: any) => t.name("schema")]),
        (_: any) => _.Board.name("X.Board").ref(true).singleton("x.Board").adjacencies({
          pins: (a: any) => a.relation([_.Pins]).me("board") }),
      ])]))]);
  };
  // Read again, a snapshot adds nothing: what an entry defines where it is linked is made where the object it is written
  // with held no such entry before (a singleton, which every store holding its schema has, or a store's own scalar).
  const boards: any = new B.Store();
  boards.transaction(pinned);
  boards.transaction(() => {
    const unnamed = [boards.Schemas.OfStr, boards.Schemas.OfInt].map((element: any) =>
      boards.Schemas.OfContainer((c: any) => c.index([boards.Schemas.OfInt]).element([element])));
    boards.X.Board(boards.singleton("x.Board"), (b: any) => b.pins((a: any) => a.insert((e: any) => e.schema(unnamed[0])).insert(
      (e: any) => e.schema(unnamed[1]))));
  });
  const pinning = Plain.ToPlain(boards).Snapshot();
  const pinnedAgain: any = new B.Store();
  pinnedAgain.transaction(pinned);
  const board = pinnedAgain.singleton("x.Board");
  assert(same(Plain.FromPlain(pinnedAgain).Snapshot(pinning).map((made) => made === board), [true])); // its data, the board
  assert(same(Plain.FromPlain(pinnedAgain).Snapshot(pinning).map((made) => made === board), [true])); // read again: found
  assert([...board.pins].length === 2);
  assert(same([...board.pins].map((e: any) => e.schema.element),
    [pinnedAgain.Schemas.OfStr.as_schema(), pinnedAgain.Schemas.OfInt.as_schema()]));

  /** Tags an object schema has, declared from its end: any object schema's, the store's own included. */
  const objectsTagged = (txn: any): any => {
    const S = txn.Schemas;
    return txn.uid("objects_tagged/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ Tags: S.OfRelation, Tag: S.OfObject, Object: S.OfObject })
      .spec([
        (_: any) => _.Tags.name("X.Tags").links([(t: any) => t.name("object"), (t: any) => t.name("tag")]),
        (_: any) => _.Tag.name("X.Tag").ref(true),
        (_: any) => _.Object.name("Schemas.OfObject").adjacencies({ tags: (a: any) => a.relation([_.Tags]).me("object") }),
      ])]))]);
  };
  // The store's own object schema, holding entries of the store's own relations, is written with its extension's alone.
  const own: any = new B.Store();
  own.transaction(objectsTagged);
  own.transaction(() => {
    own.Schemas.OfObject(own.Schemas.OfObject, (x: any) => x.tags([own.X.Tag((t: any) => t)]));
  });
  const owned = Plain.ToPlain(own).Snapshot() as any[];
  assert(same(spec_of(owned, "Schemas.OfObject", 1).filter((m) => m.has("as_adjacency")).map((m) => m.get("as_adjacency").get("name")), ["tags"]));
  const mirror: any = new B.Store();
  mirror.transaction(objectsTagged);
  Plain.FromPlain(mirror).Snapshot(owned);
  assert([...new B.OfObject.Builder(mirror.Schemas.OfObject, false, mirror).tags].length === 1
    && text(Plain.ToPlain(mirror).Snapshot()) === text(owned));
}